<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 03 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">Advertising Consolidation</div>
  <p class="doris-cover-lead">Load Google, Meta, and TikTok CSV exports, standardize their fields, remove duplicates, and combine them into one advertising detail table.</p>
  <span class="doris-cover-note">Seed · dbt_utils · View · QUALIFY · Table · Data Test</span>
</div>

## 1. Check the execution environment

Run this cell first. It uses the Demo dbt environment and current Doris connection settings, then confirms that a Backend is available.

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("Start Jupyter from the dbt-for-apache-doris repository or a subdirectory.")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 3: Advertising Consolidation

Growth teams receive daily exports from Google, Meta, and TikTok, but each platform uses different field names and may send duplicate rows. This Demo creates a consistent channel-by-day detail table for conversion analysis, campaign trends, and channel comparison.

<table class="doris-index">
  <tr><th>Business users</th><td>Growth analytics, advertising operations, and marketing data teams</td></tr>
  <tr><th>Business question</th><td>How can analysts compare clicks, impressions, views, and conversions across three advertising channels with one definition?</td></tr>
  <tr><th>Metric rule</th><td>Use source + ad_date grain; filter null dates and remove exact duplicate records</td></tr>
  <tr><th>Delivered dataset</th><td>Channel-by-day advertising detail table <code>int__ads_unified</code></td></tr>
</table>

The cells below show how CSV files enter Doris through dbt Seed, then pass through field standardization, `QUALIFY` deduplication, and consolidation.

<div class="doris-flow">
  <div class="doris-flow-step"><strong>CSV inputs</strong>Google, Meta, and TikTok files</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>dbt Seed</strong>Load as Doris Tables</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Standardize and deduplicate</strong>Align fields with <code>QUALIFY</code></div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Consolidated detail</strong><code>int__ads_unified</code></div>
</div>

### 2.1 Prepare CSV files and run dbt Seed

Google and Meta contain duplicate rows on August 1; TikTok does not. dbt Seed first loads all three CSV files into Doris Tables.

In [ ]:
ads_dir = runner.examples_root / "doris-demos/consolidate"
runner.show_file("Fixture SQL", ads_dir / "scripts/setup.sql")
runner.show_file("Google CSV", ads_dir / "seeds/googleads.csv")
runner.show_file("Meta CSV", ads_dir / "seeds/metaads.csv")
runner.show_file("TikTok CSV", ads_dir / "seeds/tiktokads.csv")
runner.run_sql_file("Create advertising Demo databases", ads_dir / "scripts/setup.sql")
runner.run_dbt("Install dbt_utils", ads_dir, "deps")
runner.run_dbt("Load three advertising Seeds", ads_dir, "seed", "--select", "googleads", "metaads", "tiktokads")
runner.query("Seed input row counts", """
select 'googleads' as table_name, count(*) as table_rows from dbt_demo_consolidate.googleads
union all select 'metaads', count(*) from dbt_demo_consolidate.metaads
union all select 'tiktokads', count(*) from dbt_demo_consolidate.tiktokads
order by table_name
""")

### 2.2 Standardize fields and deduplicate

The three staging models read the Seeds with `ref()`: Meta combines `views_1 + views_2` into `views`, TikTok maps `views_1` to `views`, and `row_number()` plus `QUALIFY` removes duplicates.

In [ ]:
for model_name in ("googleads", "metaads", "tiktokads"):
    runner.show_file(f"{model_name} staging Model", ads_dir / f"models/stg__ads_{model_name}.sql")
runner.run_dbt("Create three staging Views", ads_dir, "run", "--select", "stg__ads_googleads", "stg__ads_metaads", "stg__ads_tiktokads")
runner.query("Intermediate result: deduplicated staging", """
select 'google' as source, count(*) as rows_after_dedup from dbt_demo_consolidate.stg__ads_googleads
union all select 'meta', count(*) from dbt_demo_consolidate.stg__ads_metaads
union all select 'tiktok', count(*) from dbt_demo_consolidate.stg__ads_tiktokads
order by source
""")

### 2.3 Consolidate the channels and run the uniqueness test

The final model adds a `source` field to each staging model and combines them with `union all`. `dbt_utils.unique_combination_of_columns` checks that `source + ad_date` is unique.

In [ ]:
runner.show_file("Consolidated advertising model", ads_dir / "models/int__ads_unified.sql")
runner.show_file("Uniqueness test definition", ads_dir / "models/int__ads_unified.yml")
runner.run_dbt("Create and test the consolidated advertising table", ads_dir, "build", "--select", "int__ads_unified")
runner.query("Output: consolidated advertising detail", """
select source, ad_date, clicks, impressions, views, conversions
from dbt_demo_consolidate.int__ads_unified
order by source, ad_date
""")

### 2.4 Verify the final objects

The verifier checks that the final table has six rows and no null dates, and that the three staging objects are Views while the consolidated object is a Table.

In [ ]:
runner.run_script("Verify advertising consolidation Demo", ads_dir / "scripts/verify.sh")
runner.query("Final object types", """
select table_name, table_type
from information_schema.tables
where table_schema = 'dbt_demo_consolidate'
  and table_name in ('stg__ads_googleads', 'stg__ads_metaads', 'stg__ads_tiktokads', 'int__ads_unified')
order by table_name
""")

## Complete

The three Seeds, three standardized Views, consolidated advertising Table, and uniqueness test all passed verification.